In [6]:
import sys
print(sys.executable)

c:\Users\willi\miniconda3\envs\nf_geo\python.exe


In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
    f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT', '5432')}/{os.getenv('PG_DB')}"
)
SCHEMA = os.getenv("PG_SCHEMA")

RENAME = {"codeDepartement": "code_departement",
          "codeEICResourceObject": "code_eic",
          "codeINSEECommune": "code_insee_commune"}

def prep(path):
    df = pd.read_csv(path, sep=";", dtype={"codeDepartement": str, "codeINSEECommune": str})
    df = df.rename(columns=RENAME).rename(columns=str.lower)
    if "date_racc" in df.columns:
        df["date_racc"] = pd.to_datetime(df["date_racc"])
    return df

tables = {
    "centrales_pv": "dataset/data clean/centrales_pv_6dep.csv",
    "synthese_dep_edition": "dataset/data clean/synthese_dep_edition.csv",
    "comparaison_pvgis": "dataset/data clean/comparaison_PVGIS_empirique.csv",
}
for name, path in tables.items():
    df = prep(path)
    df.to_sql(name, engine, schema=SCHEMA, if_exists="replace", index=False)
    print(name, len(df))

with engine.begin() as conn:
    conn.execute(text(f"ALTER TABLE {SCHEMA}.synthese_dep_edition ADD PRIMARY KEY (code_departement, edition)"))

centrales_pv 32958
synthese_dep_edition 54
comparaison_pvgis 6
